# spatialprot-data Demo

This notebook demonstrates how to load and visualize spatial proteomics data using `spatialprot-data`. We use the **Lin et al. 2023** dataset (CycIF + H&E, colorectal cancer, 41 tissues).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from spatialprot_data import (
    HEImagingDataset,
    MultiplexImagingDataset,
    ComposedImagingDataset,
)

DATASET_PATH = "/mnt/aimm/scratch/datasets_v2/lin2023high"
DATASET_NAME = "lin2023high"
TISSUE_ID = "lin2023high_aijcgfvk_0000"
RESOLUTION = 1.0

## 1. Loading an H&E Dataset

In [ ]:
he_dataset = HEImagingDataset(
    name=DATASET_NAME,
    path=DATASET_PATH,
    resolution=RESOLUTION,
    crop_size=224,
)

tissue_ids = he_dataset.get_tissue_ids()
print(f"Number of tissues: {len(tissue_ids)}")
print(f"First 5 tissue IDs: {tissue_ids[:5]}")

### Visualize an H&E tissue image

We load the full tissue (without normalization) and display a center crop for visibility.

In [ ]:
he_tissue = he_dataset.get_tissue(TISSUE_ID, preprocess=False, image_mode="HWC")
he_img = he_tissue.tissue.numpy() if isinstance(he_tissue.tissue, torch.Tensor) else he_tissue.tissue
print(f"H&E image shape: {he_img.shape}, dtype: {he_img.dtype}")

# Center crop for display
H, W, C = he_img.shape
crop = 2048
cy, cx = H // 2, W // 2
he_crop = he_img[cy - crop:cy + crop, cx - crop:cx + crop]

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(he_crop.astype(np.uint8))
ax.set_title(f"H&E — {TISSUE_ID} (center {2*crop}x{2*crop} crop)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Loading a Multiplex (CycIF) Dataset

In [ ]:
mx_dataset = MultiplexImagingDataset(
    name=DATASET_NAME,
    path=DATASET_PATH,
    modality="cycif",
    normalization="identity",
    resolution=RESOLUTION,
    crop_size=224,
)

print(f"Channels ({len(mx_dataset.channel_list)}):")
print(mx_dataset.channel_list[["channel_name", "qc_pass"]].to_string(index=False))

### Visualize multiplex marker channels

We load the full tissue (unfiltered, no normalization) and display selected markers.

In [ ]:
mx_tissue = mx_dataset.get_tissue(TISSUE_ID, kind="complete", preprocess=False)
mx_img = mx_tissue.tissue.numpy() if isinstance(mx_tissue.tissue, torch.Tensor) else mx_tissue.tissue
channel_names = mx_tissue.channel_names
print(f"Multiplex image shape: {mx_img.shape}")
print(f"Channel names: {channel_names}")

# Select markers to display
markers = ["CD3e", "CD8a", "CD20", "Pan-CK", "Ki67", "SMA"]
marker_indices = [np.where(channel_names == m)[0][0] for m in markers]

# Same center crop as H&E
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, idx, name in zip(axes.ravel(), marker_indices, markers):
    channel = mx_img[idx, cy - crop:cy + crop, cx - crop:cx + crop]
    vmax = np.percentile(channel[channel > 0], 99) if (channel > 0).any() else 1.0
    ax.imshow(channel, cmap="inferno", vmin=0, vmax=vmax)
    ax.set_title(name, fontsize=13)
    ax.axis("off")

plt.suptitle(f"CycIF markers — {TISSUE_ID}", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Side-by-side: H&E and multiplex

Compare the same tissue region across modalities.

In [ ]:
# Composite: Pan-CK (green) + CD3e (red) + DAPI/Hoechst (blue)
hoechst_idx = np.where(channel_names == "Hoechst")[0][0]
panck_idx = np.where(channel_names == "Pan-CK")[0][0]
cd3e_idx = np.where(channel_names == "CD3e")[0][0]

region = np.s_[cy - crop:cy + crop, cx - crop:cx + crop]

def norm_channel(ch):
    q99 = np.percentile(ch[ch > 0], 99) if (ch > 0).any() else 1.0
    return np.clip(ch / max(q99, 1e-6), 0, 1)

composite = np.stack([
    norm_channel(mx_img[cd3e_idx][region]),   # R: CD3e (T cells)
    norm_channel(mx_img[panck_idx][region]),   # G: Pan-CK (epithelial)
    norm_channel(mx_img[hoechst_idx][region]), # B: Hoechst (nuclei)
], axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(he_crop.astype(np.uint8))
axes[0].set_title("H&E", fontsize=14)
axes[0].axis("off")

axes[1].imshow(composite)
axes[1].set_title("CycIF composite (R: CD3e, G: Pan-CK, B: Hoechst)", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 3. Composed Multi-modal Dataset

In [ ]:
composed = ComposedImagingDataset(
    name=DATASET_NAME,
    path=DATASET_PATH,
    modalities=["he", "cycif"],
    resolution=RESOLUTION,
    crop_size=224,
    modality_kwargs={"cycif": {"normalization": "identity"}},
)

print(f"Available modalities: {composed.get_available_modalities()}")
print(f"Total tissues: {len(composed.get_tissue_ids())}")
print(f"Modalities for {TISSUE_ID}: {composed.get_modalities_of_tissue(TISSUE_ID)}")

In [ ]:
# Load both modalities at once via the composed interface
composed_tissue = composed.get_composed_tissue(TISSUE_ID, kind="complete", preprocess=False)
print(f"Composed tissue modalities: {list(composed_tissue.modalities.keys())}")
for mod, tissue in composed_tissue.modalities.items():
    print(f"  {mod}: shape={tissue.tissue.shape}")

## 4. Tissue Masks

In [ ]:
tissue_mask = he_dataset.get_tissue_mask(TISSUE_ID)
mask = tissue_mask.mask

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(he_crop.astype(np.uint8))
axes[0].set_title("H&E", fontsize=13)
axes[0].axis("off")

mask_crop = mask[cy - crop:cy + crop, cx - crop:cx + crop]
axes[1].imshow(mask_crop, cmap="gray")
axes[1].set_title("Tissue mask", fontsize=13)
axes[1].axis("off")

plt.tight_layout()
plt.show()
print(f"Mask shape: {mask.shape}, tissue coverage: {mask.mean():.1%}")

## 5. Channel Metadata

In [ ]:
# Channel list with QC status and UniProt IDs
mx_dataset.channel_list

In [ ]:
# Per-tissue channel availability (first 5 tissues, first 8 channels)
mx_dataset.image_channel_map.iloc[:5, :8]